In [ ]:
# Import custom scripts
from scripts import analysis, datahandler, visuals, simulation
from importlib import reload

for module in (analysis, datahandler, visuals, simulation):
    reload(module)

# Import python essentials
import os
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
root = "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/test"

data = []
for tp in os.listdir(root):
    data_dir = os.path.join(root, tp, "data")
    if not os.path.isdir(data_dir):
        continue
    for proj in os.listdir(data_dir):
        folder = os.path.join(data_dir, proj)
        if not os.path.isdir(folder):
            continue
        s_val = datahandler.load_array("S-order_2dcurved_r-40.0um", folderpath=folder)
        data.append({
            "Time": int(tp.split("h")[0]),
            "Projection": proj.split("proj_")[-1],
            "S": s_val
        })

# Convert to dataframe
df = pd.DataFrame(data)
df = df.explode("S")
df["S"] = df["S"].astype(float)

# Sort times numerically
df["Time"] = df["Time"].astype(float)
df = df.sort_values("Time")

# Violin plot grouped by projection layer
plt.figure(figsize=(10, 5))
sns.violinplot(
    data=df,
    x="Time",
    y="S",
    hue="Projection",
    inner="quartile",
    cut=0,
    linewidth=1
)
plt.title("Nematic order $S$ over Time per Projection Layer")
plt.xlabel("Time (h)")
plt.ylabel("nematic order $S$")
plt.legend(title="Projection", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(root, "S-order over Time per Projection Layer"), dpi=300)
plt.show()

# TAC figures

In [ ]:
# =================================================================
local_nneigh = 7
crop_max_ar = 5  # Max for colorbar !
local_S_weighted_max = 3.0  # Max for colorbar !


# =================================================================

def quick_load_seg(path, xy_pixel_scale):
    seg_ellips_results = datahandler.load_array(name=f"{0 + 1:04d}_seg-ellipses", folderpath=path,
                                                return_df=True)
    # seg_ellips_results = datahandler.load_array(name=f"{img_name}_z{z_sel + 1}_cp_masks_ellipses",
    #                                             folderpath=path, return_df=True)
    # os.path.join(os.path.dirname(img_path),"z_slice_segmentation", img_name)
    seg_ellips_results["Ellipse.Orientation"] *= -1
    seg_ellips_results["Ellipse.Radius1"] *= xy_pixel_scale
    seg_ellips_results["Ellipse.Radius2"] *= xy_pixel_scale
    seg_ellips_results["aspect_ratio"] = seg_ellips_results["Ellipse.Radius1"] / seg_ellips_results[
        "Ellipse.Radius2"]
    seg_ellips_results["shape_scalar"] = seg_ellips_results["aspect_ratio"] - 1
    seg_ellips_results = seg_ellips_results[seg_ellips_results["aspect_ratio"] < crop_max_ar]
    return seg_ellips_results


img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/segmentation_examples/72.tif"
print(f"Selected image path: {img_path}")
img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=0, c_sel_idx=0, reduce_xy=1,
                                     reduce_z=1, norm_vals=False, custom_scaling=None, custom_unit="um")
img_raw, img_dim, img_scale, img_unit = img_load
resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
resfig_dir_2dsliced = os.path.join(resfig_dir, "2d-sliced_analysis")
resdata_dir_2dsliced = os.path.join(resdata_dir, "2d-sliced_analysis")
datahandler.create_dir(resfig_dir_2dsliced)
datahandler.create_dir(resdata_dir_2dsliced)
seg_ellips_results_72h = quick_load_seg(path=resdata_dir_2dsliced, xy_pixel_scale=np.mean(img_scale[1:]))
print(seg_ellips_results_72h["Ellipse.Radius1"].max())
medial_curve_72h = datahandler.load_array("curve", folderpath=resdata_dir_2dsliced)
seg_directors_2d_72h = np.column_stack((seg_ellips_results_72h["Ellipse.Center.X"],
                                        seg_ellips_results_72h["Ellipse.Center.Y"],
                                        np.cos(np.radians(seg_ellips_results_72h["Ellipse.Orientation"])),
                                        np.sin(np.radians(seg_ellips_results_72h["Ellipse.Orientation"]))))
seg_directors_2d_72h[:, :2] *= img_scale[1:]
s_parallel_72h, s_orthogonal_72h = analysis.proj2curve(seg_directors_2d_72h[:, :2], medial_curve_72h)

local_idxs_72h = analysis.coord_search_neighbours(seg_directors_2d_72h[:, :2], k=local_nneigh)
S_2d_seg_local_72h, n_2d_seg_local_72h = analysis.avg_2d_nem_tens(seg_directors_2d_72h, weights=None,
                                                                  neigh_idxs=local_idxs_72h)
S_2d_seg_local_weighted_72h, n_2d_seg_local_72h = analysis.avg_2d_nem_tens(seg_directors_2d_72h, local_idxs_72h,
                                                                           weights=np.array(
                                                                               seg_ellips_results_72h["shape_scalar"]))

img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/segmentation_examples/96.tif"
print(f"Selected image path: {img_path}")
img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=0, c_sel_idx=0, reduce_xy=1,
                                     reduce_z=1, norm_vals=False, custom_scaling=None, custom_unit="um")
img_raw, img_dim, img_scale, img_unit = img_load
resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
resfig_dir_2dsliced = os.path.join(resfig_dir, "2d-sliced_analysis")
resdata_dir_2dsliced = os.path.join(resdata_dir, "2d-sliced_analysis")
datahandler.create_dir(resfig_dir_2dsliced)
datahandler.create_dir(resdata_dir_2dsliced)
seg_ellips_results_96h = quick_load_seg(path=resdata_dir_2dsliced, xy_pixel_scale=np.mean(img_scale[1:]))
print(seg_ellips_results_96h["Ellipse.Radius1"].max())
medial_curve_96h = datahandler.load_array("curve", folderpath=resdata_dir_2dsliced)
seg_directors_2d_96h = np.column_stack((seg_ellips_results_96h["Ellipse.Center.X"],
                                        seg_ellips_results_96h["Ellipse.Center.Y"],
                                        np.cos(np.radians(seg_ellips_results_96h["Ellipse.Orientation"])),
                                        np.sin(np.radians(seg_ellips_results_96h["Ellipse.Orientation"]))))
seg_directors_2d_96h[:, :2] *= img_scale[1:]
s_parallel_96h, s_orthogonal_96h = analysis.proj2curve(seg_directors_2d_96h[:, :2], medial_curve_96h)
local_idxs_96h = analysis.coord_search_neighbours(seg_directors_2d_96h[:, :2], k=local_nneigh)
S_2d_seg_local_96h, n_2d_seg_local_96h = analysis.avg_2d_nem_tens(seg_directors_2d_96h, weights=None,
                                                                  neigh_idxs=local_idxs_96h)
S_2d_seg_local_weighted_96h, n_2d_seg_local_96h = analysis.avg_2d_nem_tens(seg_directors_2d_96h, local_idxs_96h,
                                                                           weights=np.array(
                                                                               seg_ellips_results_96h["shape_scalar"]))

img_path = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/segmentation_examples/120.tif"
print(f"Selected image path: {img_path}")
img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=0, c_sel_idx=0, reduce_xy=1,
                                     reduce_z=1, norm_vals=False, custom_scaling=None, custom_unit="um")
img_raw, img_dim, img_scale, img_unit = img_load
resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
resfig_dir_2dsliced = os.path.join(resfig_dir, "2d-sliced_analysis")
resdata_dir_2dsliced = os.path.join(resdata_dir, "2d-sliced_analysis")
datahandler.create_dir(resfig_dir_2dsliced)
datahandler.create_dir(resdata_dir_2dsliced)
seg_ellips_results_120h = quick_load_seg(path=resdata_dir_2dsliced, xy_pixel_scale=np.mean(img_scale[1:]))
medial_curve_120h = datahandler.load_array("curve", folderpath=resdata_dir_2dsliced)
seg_directors_2d_120h = np.column_stack((seg_ellips_results_120h["Ellipse.Center.X"],
                                         seg_ellips_results_120h["Ellipse.Center.Y"],
                                         np.cos(np.radians(seg_ellips_results_120h["Ellipse.Orientation"])),
                                         np.sin(np.radians(seg_ellips_results_120h["Ellipse.Orientation"]))))
seg_directors_2d_120h[:, :2] *= img_scale[1:]
s_parallel_120h, s_orthogonal_120h = analysis.proj2curve(seg_directors_2d_120h[:, :2], medial_curve_120h)
local_idxs_120h = analysis.coord_search_neighbours(seg_directors_2d_120h[:, :2], k=local_nneigh)
S_2d_seg_local_120h, n_2d_seg_local_120h = analysis.avg_2d_nem_tens(seg_directors_2d_120h, weights=None,
                                                                    neigh_idxs=local_idxs_120h)
S_2d_seg_local_weighted_120h, n_2d_seg_local_120h = analysis.avg_2d_nem_tens(seg_directors_2d_120h, local_idxs_120h,
                                                                             weights=np.array(
                                                                                 seg_ellips_results_120h[
                                                                                     "shape_scalar"]))

In [ ]:
figpath = "/Users/andreadi/Library/CloudStorage/OneDrive-UniversitédeGenève/Academic/Data/1_gastruloid/segmentation_examples/!COMPARISON"

visuals.plot_violinplot_comparative(
    sequence_data=[[seg_ellips_results_72h["Ellipse.Radius1"], seg_ellips_results_72h["Ellipse.Radius2"]],
                   [seg_ellips_results_96h["Ellipse.Radius1"], seg_ellips_results_96h["Ellipse.Radius2"]],
                   [seg_ellips_results_120h["Ellipse.Radius1"], seg_ellips_results_120h["Ellipse.Radius2"]]],
    sequence_labels=["72h", "96h", "120h"], group_labels=["Long Axis", "Short Axis"],
    title=f"Ellipse Axes ({img_unit})", savefig=os.path.join(figpath, "ellipse-axes.pdf"))

visuals.plot_violinplot_comparative(
    sequence_data=[[S_2d_seg_local_72h, S_2d_seg_local_weighted_72h],
                   [S_2d_seg_local_96h, S_2d_seg_local_weighted_96h],
                   [S_2d_seg_local_120h, S_2d_seg_local_weighted_120h]],
    sequence_labels=["72h", "96h", "120h"], group_labels=["Normalised", "Weighted by AR-1"],
    title=f"Local Nematic Order w.r.t. {local_nneigh - 1} Neighbours",
    savefig=os.path.join(figpath, f"local-nematic-order-k={local_nneigh}.pdf"))

# visuals.plot_violinplot(sequence_data=[seg_ellips_results_72h["aspect_ratio"], seg_ellips_results_96h["aspect_ratio"],
#                                        seg_ellips_results_120h["aspect_ratio"]], sequence_labels=["72h", "96h", "120h"],
#                         title=f"Aspect Ratio ({img_unit})", savefig=os.path.join(figpath, "ellipse-aspect-ratios.pdf"), figsize=(5,5))
#
# sequence_labels = ["72h", "96h", "120h"]
#
# plt.figure(figsize=(6, 5))
# # Plot the lines with markers, colors, and linewidths
# plt.plot([s_parallel_72h.max(), s_parallel_96h.max(), s_parallel_120h.max()],
#          marker='s', linestyle='-', color='tab:blue', linewidth=2, markersize=8,
#          label=f"Parallel to elongation")
#
# plt.plot([s_orthogonal_72h.max(), s_orthogonal_96h.max(), s_orthogonal_120h.max()],
#          marker='o', linestyle='-', color='tab:orange', linewidth=2, markersize=8,
#          label=f"Orthogonal to elongation")
#
# # X-axis labels
# plt.xticks(range(len(sequence_labels)), sequence_labels)
#
# # Labels and title
# plt.xlabel("Time")
# plt.ylabel(f"Elongation ({img_unit})")
# plt.title(f"Tissue Elongation over Time ({img_unit})")
#
# # Grid and legend
# plt.grid(alpha=0.3)
# plt.legend()
# plt.savefig(os.path.join(figpath, "tissue-elongation.pdf"))
# plt.tight_layout()
# plt.show()


# Anàlisi del turbo run

In [ ]:
def extract_all_segmentations(chosen_time_point, chosen_size, chosen_gastruloid, local_radius=20, crop_max_ar=5):
    img_path = f"/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER/{chosen_time_point}/{chosen_size}/{chosen_gastruloid}.tif"
    print(f"Selected image path: {img_path}")
    if not os.path.exists(img_path):
        print(f"Image path does not exist: {img_path} !")
        return None
    img_scale = analysis.load_img_scaling(img_path)
    img_name = os.path.basename(img_path).split(".tif")[0]
    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
    resfig_dir_2dsliced = os.path.join(resfig_dir, "2d-sliced_analysis")
    if not os.path.exists(resfig_dir_2dsliced):
        datahandler.create_dir(resfig_dir_2dsliced)
    resdata_dir_2dsliced = os.path.join(resdata_dir, "2d-sliced_analysis")
    if not os.path.exists(resdata_dir_2dsliced):
        datahandler.create_dir(resdata_dir_2dsliced)

    seg_path = os.path.join(os.path.dirname(img_path), "z_slice_segmentation", img_name)
    if not os.path.exists(seg_path):
        print(f"No segmentation for {img_name} !")
        return None
    seg_items = os.listdir(seg_path)
    seg_masks = np.array([f for f in seg_items if f.endswith(".png")])
    seg_z_vals = np.array([
        int(m.group(1)) for f in seg_masks if (m := re.search(r"_z(\d+)_", f))
    ])
    valid_z_all = []
    aspect_ratio_all = []
    S_local_normalised_all = []
    S_local_weighted_all = []
    S_avg_normalised_all = []
    S_avg_weighted_all = []
    theta_all = []
    xpos_all = []
    ypos_all = []
    radius1_all = []
    radius2_all = []
    for chosen_zslice in seg_z_vals:
        try:
            seg_ellips_results = datahandler.load_array(name=f"{img_name}_z{chosen_zslice}_cp_masks_ellipses",
                                                        folderpath=seg_path, return_df=True, debug=False)
        except:
            print(f"[!!] Fatal error when loading ellipses for z={chosen_zslice}")
            continue
        if seg_ellips_results is None:
            print(f"No ellipses found for z={chosen_zslice}")
            continue
        seg_ellips_results["Ellipse.Orientation"] *= -1
        seg_ellips_results["Ellipse.Radius1"] *= np.mean(img_scale[1:])
        seg_ellips_results["Ellipse.Radius2"] *= np.mean(img_scale[1:])
        seg_ellips_results["aspect_ratio"] = seg_ellips_results["Ellipse.Radius1"] / seg_ellips_results[
            "Ellipse.Radius2"]
        seg_ellips_results["shape_scalar"] = seg_ellips_results["aspect_ratio"] - 1
        seg_ellips_results = seg_ellips_results[seg_ellips_results["aspect_ratio"] < crop_max_ar]
        seg_directors_2d = np.column_stack((seg_ellips_results["Ellipse.Center.X"],
                                            seg_ellips_results["Ellipse.Center.Y"],
                                            np.cos(np.radians(seg_ellips_results["Ellipse.Orientation"])),
                                            np.sin(np.radians(seg_ellips_results["Ellipse.Orientation"]))))
        seg_directors_2d[:, :2] *= img_scale[1:]
        if len(seg_directors_2d) > 1:
            local_idxs = analysis.coord_search_radius(seg_directors_2d[:, :2], r=local_radius)
            reload(analysis)
            S_2d_seg_local, _ = analysis.avg_2d_nem_tens(seg_directors_2d, weights=None,
                                                         neigh_idxs=local_idxs)
            S_2d_seg_local_weighted, _ = analysis.avg_2d_nem_tens(seg_directors_2d, local_idxs,
                                                                  weights=np.array(seg_ellips_results[
                                                                                       "shape_scalar"]))
            slice_idxs = [np.arange(len(seg_directors_2d))]
            S_2d_seg_slice, _ = analysis.avg_2d_nem_tens(seg_directors_2d,
                                                         neigh_idxs=slice_idxs, weights=None)
            S_2d_seg_slice_weighted, _ = analysis.avg_2d_nem_tens(seg_directors_2d,
                                                                  neigh_idxs=slice_idxs,
                                                                  weights=np.array(seg_ellips_results["shape_scalar"]))
            valid_z_all.append(chosen_zslice)
            aspect_ratio_all.append(seg_ellips_results["aspect_ratio"].to_numpy())
            S_local_normalised_all.append(S_2d_seg_local)
            S_local_weighted_all.append(S_2d_seg_local_weighted)
            S_avg_normalised_all.append(S_2d_seg_slice[0])
            S_avg_weighted_all.append(S_2d_seg_slice_weighted[0])
            theta_all.append(np.array(seg_ellips_results["Ellipse.Orientation"]))
            xpos_all.append(np.array(seg_ellips_results["Ellipse.Center.X"]))
            ypos_all.append(np.array(seg_ellips_results["Ellipse.Center.Y"]))
            radius1_all.append(np.array(seg_ellips_results["Ellipse.Radius1"]))
            radius2_all.append(np.array(seg_ellips_results["Ellipse.Radius2"]))
    return img_path, img_scale, valid_z_all, aspect_ratio_all, S_local_normalised_all, S_local_weighted_all, S_avg_normalised_all, S_avg_weighted_all, theta_all, xpos_all, ypos_all, radius1_all, radius2_all

In [ ]:
exclusion_list = [["72", "200", "Gas4"],
                  ["72", "300", "Gas4"],
                  ["96", "300", "Gas9"],
                  ["104", "300", "Gas6"],
                  ["104", "300", "Gas7"],
                  ["112", "200", "Gas4"],
                  ["112", "200", "Gas5"],
                  ["112", "300", "Gas 5"],
                  ["112", "300", "Gas 8ab"],
                  ["120", "200", "Gas2"],
                  ["120", "200", "Gas3"],
                  ["120", "200", "Gas4"],
                  ["120", "300", "Gas5"],
                  ["120", "300", "Gas7"],
                  ["120", "300", "Gas11"]]
root_dir = "/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER/"
time_points = ["72", "96", "104", "112", "120"]
seeding_sizes = ["200", "300"]
all_data = {}
for tp in time_points:
    print(f">> Processing time: {tp}")
    all_data[tp] = {}
    for size in seeding_sizes:
        print(f">>>> Processing seeding size: {size}")
        folder_path = os.path.join(root_dir, tp, size)
        if not os.path.exists(folder_path):
            continue
        file_names = os.listdir(folder_path)
        gastruloid_names = [f.split(".tif")[0] for f in file_names if f.endswith(".tif")]
        all_data[tp][size] = {}
        for gas_name in gastruloid_names:
            if [tp, size, gas_name] in exclusion_list:
                print(f"!!! Skipping {gas_name} (excluded)")
                continue
            analysis_packet = extract_all_segmentations(chosen_time_point=tp, chosen_size=size,
                                                        chosen_gastruloid=gas_name)
            if analysis_packet is not None:
                all_data[tp][size][gas_name] = analysis_packet

# Dataset Creation

In [ ]:
# ---- Local (cell-level) ----
records = []

for tp, sizes in all_data.items():
    for size, gastruloids in sizes.items():
        for gas_name, packet in gastruloids.items():
            img_path, img_scale, valid_z_all, aspect_ratio_all, S_local_normalised_all, S_local_weighted_all, \
                S_avg_normalised_all, S_avg_weighted_all, theta_all, xpos_all, ypos_all, radius1_all, radius2_all = packet

            for z, aspect_array, S_arr, S_norm_arr, theta_array, xpos_array, ypos_array, radius1_array, radius2_array in zip(
                    valid_z_all, aspect_ratio_all, S_local_weighted_all, S_local_normalised_all, theta_all,
                    xpos_all, ypos_all, radius1_all, radius2_all):
                for S_val, S_norm_val, ar, th, xpos, ypos, radius1, radius2 in zip(
                        S_arr, S_norm_arr, aspect_array, theta_array, xpos_array, ypos_array, radius1_array,
                        radius2_array):
                    records.append({
                        "img_path": img_path,
                        "img_scale": img_scale,
                        "time_point": tp,
                        "size": size,
                        "gastruloid": gas_name,
                        "slice_z": z,
                        "aspect_ratio": ar,
                        "S_value": S_val,
                        "S_value_norm": S_norm_val,
                        "theta": th,
                        "xpos": xpos,
                        "ypos": ypos,
                        "radius1": radius1,
                        "radius2": radius2
                    })

local_df = pd.DataFrame.from_records(records)

# ---- Slice-level (average) ----
avg_records = []

for tp, sizes in all_data.items():
    for size, gastruloids in sizes.items():
        for gas_name, packet in gastruloids.items():
            img_path, img_scale, valid_z_all, aspect_ratio_all, S_local_normalised_all, S_local_weighted_all, \
                S_avg_normalised_all, S_avg_weighted_all, theta_all, xpos_all, ypos_all, radius1_all, radius2_all = packet

            for z, aspect_array, S_val, S_norm_val, theta_array, radius1_array, radius2_array in zip(
                    valid_z_all, aspect_ratio_all, S_avg_weighted_all, S_avg_normalised_all, theta_all,
                    radius1_all, radius2_all):
                avg_records.append({
                    "img_path": img_path,
                    "img_scale": img_scale,
                    "time_point": tp,
                    "size": size,
                    "gastruloid": gas_name,
                    "slice_z": z,
                    "aspect_ratio_avg": np.mean(aspect_array),
                    "S_avg": S_val,
                    "S_avg_norm": S_norm_val,
                    "theta_avg": np.mean(theta_array),
                    "radius1_avg": np.mean(radius1_array),
                    "radius2_avg": np.mean(radius2_array)
                })

avg_df = pd.DataFrame.from_records(avg_records)

In [ ]:
num_gastruloids = np.sum(np.array(local_df.groupby(["time_point", "size"])["gastruloid"].nunique()))
num_sizes = avg_df['size'].nunique()
all_sizes = avg_df["size"].unique()
num_time_points = avg_df['time_point'].nunique()
all_time_points = avg_df['time_point'].unique()
print(f"Total number of gastruloids: {num_gastruloids}")
print(f"Initial seeding sizes: {num_sizes}, {all_sizes} in number of cells")
print(f"Time points: {num_time_points}, {all_time_points} (hours post-seeding)")
print(
    f"Columns of local z-slice dataframe: \n {list(local_df.columns)}: \n and average dataframe: \n {list(avg_df.columns)}")

In [ ]:
# define the filter
filter_condition = (
        (avg_df["time_point"] == "120") &
        (avg_df["size"] == "300") &
        (avg_df["gastruloid"] == "Gas10") &
        (avg_df["slice_z"] == 38)
)

# Remove from avg_df
avg_df = avg_df[~filter_condition].reset_index(drop=True)

# Remove the same slice from local_df
local_df = local_df[
    ~(
            (local_df["time_point"] == "120") &
            (local_df["size"] == "300") &
            (local_df["gastruloid"] == "Gas10") &
            (local_df["slice_z"] == 38)
    )
].reset_index(drop=True)

print("Slice removed from both dataframes.")

# -- SAVE Dataset --

In [ ]:
local_df.to_feather(
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/local_df_backup.feather")
avg_df.to_feather(
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/avg_df_backup.feather")

print("Feather backups saved safely!")

# -- LOAD Dataset --

In [ ]:
local_df = pd.read_feather(
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/local_df_backup.feather")
print(f"Loaded local df with columns {list(local_df.columns)}")
avg_df = pd.read_feather(
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/avg_df_backup.feather")
print(f"Loaded average df with columns {list(avg_df.columns)}")

# Plotting

In [ ]:
def plot_violin_with_swarm(df, y_col, title=None, palette='Set2', show_swarm=False, swarm_color='k', swarm_alpha=0.5,
                           swarm_size=3):
    df = df.copy()
    df["time_point"] = pd.Categorical(
        df["time_point"],
        categories=sorted(df["time_point"].unique(), key=int),
        ordered=True
    )

    plt.figure(figsize=(12, 6))
    sns.violinplot(
        data=df,
        x='time_point',
        y=y_col,
        hue='size',
        split=True,
        palette=palette, inner="quartile",
    )

    if show_swarm:
        sns.swarmplot(
            data=df,
            x='time_point',
            y=y_col,
            hue='size',
            dodge=True,
            color=swarm_color,
            alpha=swarm_alpha,
            size=swarm_size, legend=False
        )

    plt.xlabel("Time point")
    plt.ylabel(y_col.replace('_', ' ').capitalize())
    if title is not None:
        plt.title(title)
    plt.grid(alpha=0.3, axis='y')
    plt.legend(title="Size")  # keep the violinplot legend
    plt.savefig(os.path.join(comp_fig_dir, f'{title}.{comp_fig_type}'), dpi=comp_fig_dpi)
    plt.show()


def plot_aligned_z(
        df,
        y_col,
        time_point=None,
        size=None,
        group_cols=None,
        hue_col="time_point",
        style_col="size",
        mode="line",  # "line" or "scatter"
        separate_gastruloids=False,
        estimator="mean",
        ci="sd",
        markers=True,
        dashes=False,
        alpha=0.7,
        s=40,
        figsize=(12, 6)
):
    if group_cols is None:
        group_cols = ["time_point", "size", "gastruloid"]

    df_copy = df.copy()

    # ---- Apply filters ----
    if time_point is not None:
        df_copy = df_copy[df_copy["time_point"] == str(time_point)]
    if size is not None:
        df_copy = df_copy[df_copy["size"] == str(size)]

    # ---- Center z by gastruloid ----
    df_copy["z_centered"] = df_copy.groupby(group_cols)["slice_z"] \
        .transform(lambda x: x - np.median(x))

    # ---- Plot setup ----
    plt.figure(figsize=figsize)

    if separate_gastruloids:
        g = sns.FacetGrid(
            df_copy,
            col="gastruloid",
            hue=hue_col,
            col_wrap=4,
            sharex=True,
            sharey=True,
            height=3
        )
        if mode == "line":
            # 🔧 Removed `style` to prevent label duplication error
            g.map_dataframe(
                sns.lineplot,
                x="z_centered", y=y_col,
                estimator=estimator,
                ci=ci,
                markers=markers,
                dashes=dashes
            )
        else:
            g.map_dataframe(
                sns.scatterplot,
                x="z_centered", y=y_col,
                alpha=alpha,
                s=s
            )

        g.add_legend()
        g.set_titles(col_template="{col_name}")
        g.set_axis_labels("Slice z (centered)", y_col)
        plt.subplots_adjust(top=0.9)
        suptitle = f"Aligned {y_col} vs z — separated by gastruloid"
        g.fig.suptitle(suptitle)
        plt.savefig(os.path.join(comp_fig_dir, f'{suptitle}_time-{time_point}_size={size}.{comp_fig_type}'),
                    dpi=comp_fig_dpi)
        plt.show()
    else:
        if mode == "line":
            sns.lineplot(
                data=df_copy,
                x="z_centered",
                y=y_col,
                hue=hue_col,
                style=style_col,
                estimator=estimator,
                ci=ci,
                markers=markers,
                dashes=dashes
            )
        elif mode == "scatter":
            sns.scatterplot(
                data=df_copy,
                x="z_centered",
                y=y_col,
                hue=hue_col,
                style=style_col,
                alpha=alpha,
                s=s
            )

        plt.xlabel("Slice z (centered on median per gastruloid)")
        plt.ylabel(y_col)
        title = f"Aligned {y_col} vs z"
        if time_point:
            title += f" — time {time_point}"
        if size:
            title += f", size {size}"
        plt.title(title)
        plt.grid(alpha=0.3)
        plt.savefig(os.path.join(comp_fig_dir, f'{title}_time-{time_point}_size={size}.{comp_fig_type}'),
                    dpi=comp_fig_dpi)
        plt.show()


def crop_z_fraction(df, z_min_frac=0.0, z_max_frac=1.0):
    df = df.copy()
    cropped = []

    for (_, sub) in df.groupby(["time_point", "size", "gastruloid"]):
        unique_z = np.sort(sub["slice_z"].unique())
        n = len(unique_z)
        if n == 0:
            continue

        # Convert fractions → indices
        lo = int(np.floor(n * z_min_frac))
        hi = int(np.ceil(n * z_max_frac))

        keep_z = unique_z[lo:hi]
        cropped.append(sub[sub["slice_z"].isin(keep_z)])

    return pd.concat(cropped, ignore_index=True)


def plot_jaja(df):
    var_tp_size = (
        df.groupby(["time_point", "size", "slice_z"])["aspect_ratio"]
        .var()
        .reset_index(name="aspect_ratio_var")
    )

    var_tp_size["time_point"] = pd.Categorical(
        var_tp_size["time_point"],
        categories=sorted(var_tp_size["time_point"].unique(), key=int),
        ordered=True
    )
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=var_tp_size,
        x="time_point",
        y="aspect_ratio_var",
        hue="size",
        marker="o"
    )
    plt.title("Aspect-ratio variance (df) over time")
    plt.ylabel("Variance of aspect ratio")
    plt.grid(alpha=0.3)
    plt.show()


def middle_z_mean(df):
    """Compute mean cell count in middle 40–60% z-range per gastruloid."""
    z_sorted = df.sort_values('slice_z')
    n = len(z_sorted)
    if n < 3:
        return np.nan
    start = int(0.3 * n)
    end = int(0.7 * n)
    middle_section = z_sorted.iloc[start:end]
    return middle_section['n_cells'].mean()

In [ ]:
comp_fig_dir = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/seg-batch-analysis'
# comp_fig_dir = ""
comp_fig_dpi = 200
comp_fig_type = "png"

In [ ]:
t_sel, size_sel, gas_name = "96", "300", "Gas3"
df_time_size_gas = local_df[(local_df["time_point"] == t_sel) & (local_df["size"] == size_sel) & (
        local_df["gastruloid"] == gas_name)].sort_values("slice_z").copy()

plt.figure()
sns.lineplot(data=df_time_size_gas, x="slice_z", y="theta", marker="o")
plt.title(f"Time point {t_sel}h size {size_sel} gastruloid {gas_name}")
plt.show()
#
# # --- Compute Omega ---
# z_vals = df_time_size_gas["slice_z"].to_numpy()
# dz = np.diff(z_vals)
# dtheta = np.diff(df_time_size_gas["theta"].to_numpy())
# with np.errstate(divide='ignore', invalid='ignore'):
#     Omega = np.concatenate([[np.nan], dtheta / dz])
#
# df_time_size_gas["Omega"] = Omega
# plt.figure(figsize=(8, 4))
# sns.lineplot(df_time_size_gas, x="slice_z", y="Omega", marker="o")
# plt.title(f"Ω = dθ/dz — {t_sel}h, size {size_sel}, gastruloid {gas_name}")
# plt.xlabel("z slice")
# plt.ylabel("Ω (deg / z-unit)")
# plt.grid(alpha=0.3)
# plt.show()

In [ ]:
t_sel, size_sel, gas_name = "72", "300", "Gas1"
df_time_size_gas = local_df[(local_df["time_point"] == t_sel) & (local_df["size"] == size_sel) & (
        local_df["gastruloid"] == gas_name)].sort_values("slice_z").copy()

plt.figure()
sns.lineplot(data=df_time_size_gas, x="slice_z", y="theta", marker="o")
plt.title(f"Time point {t_sel}h size {size_sel} gastruloid {gas_name}")
plt.show()

t_sel, size_sel, gas_name = "96", "300", "Gas3"
df_time_size_gas = local_df[(local_df["time_point"] == t_sel) & (local_df["size"] == size_sel) & (
        local_df["gastruloid"] == gas_name)].sort_values("slice_z").copy()

plt.figure()
sns.lineplot(data=df_time_size_gas, x="slice_z", y="theta", marker="o")
plt.title(f"Time point {t_sel}h size {size_sel} gastruloid {gas_name}")
plt.show()

t_sel, size_sel, gas_name = "104", "300", "Gas1"
df_time_size_gas = local_df[(local_df["time_point"] == t_sel) & (local_df["size"] == size_sel) & (
        local_df["gastruloid"] == gas_name)].sort_values("slice_z").copy()

plt.figure()
sns.lineplot(data=df_time_size_gas, x="slice_z", y="theta", marker="o")
plt.title(f"Time point {t_sel}h size {size_sel} gastruloid {gas_name}")
plt.show()

t_sel, size_sel, gas_name = "112", "300", "Gas2"
df_time_size_gas = local_df[(local_df["time_point"] == t_sel) & (local_df["size"] == size_sel) & (
        local_df["gastruloid"] == gas_name)].sort_values("slice_z").copy()

plt.figure()
sns.lineplot(data=df_time_size_gas, x="slice_z", y="theta", marker="o")
plt.title(f"Time point {t_sel}h size {size_sel} gastruloid {gas_name}")
plt.show()

t_sel, size_sel, gas_name = "120", "300", "Gas9"
df_time_size_gas = local_df[(local_df["time_point"] == t_sel) & (local_df["size"] == size_sel) & (
        local_df["gastruloid"] == gas_name)].sort_values("slice_z").copy()

plt.figure()
sns.lineplot(data=df_time_size_gas, x="slice_z", y="theta", marker="o")
plt.title(f"Time point {t_sel}h size {size_sel} gastruloid {gas_name}")
plt.show()

In [ ]:
local_df_sortedtime = local_df.copy()
local_df_sortedtime["time_point"] = pd.Categorical(
    local_df_sortedtime["time_point"],
    categories=sorted(local_df_sortedtime["time_point"].unique(), key=int),
    ordered=True
)
plt.figure(figsize=(12, 6))
sns.lineplot(data=local_df_sortedtime, x="slice_z", y="theta", hue="time_point", style="size",
             marker="o", alpha=0.7, palette="coolwarm")
plt.show()

In [ ]:
local_df_sortedtime = local_df.copy()
local_df_sortedtime["time_point"] = pd.Categorical(
    local_df_sortedtime["time_point"],
    categories=sorted(local_df_sortedtime["time_point"].unique(), key=int),
    ordered=True
)
plt.figure(figsize=(12, 6))
sns.scatterplot(data=local_df_sortedtime, x="slice_z", y="theta", hue="time_point", style="size",
                marker="o", alpha=0.7, palette="coolwarm")
plt.show()

In [ ]:
local_df_sortedtime = local_df.copy()
local_df_sortedtime["time_point"] = pd.Categorical(
    local_df_sortedtime["time_point"],
    categories=sorted(local_df_sortedtime["time_point"].unique(), key=int),
    ordered=True
)
plt.figure(figsize=(12, 6))
sns.scatterplot(data=local_df_sortedtime, x="slice_z", y="aspect_ratio", hue="time_point", style="size",
                marker="o", alpha=0.7, palette="coolwarm")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=local_df_sortedtime, x="slice_z", y="S_value", hue="time_point", style="size",
             marker="o", alpha=0.7, palette="coolwarm")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=local_df_sortedtime, x="slice_z", y="aspect_ratio", hue="time_point", style="size",
             marker="o", alpha=0.7, palette="coolwarm")
plt.show()

In [ ]:
plt.figure()
sns.lineplot(local_df, x="time_point", y="aspect_ratio", hue="size")
plt.show()
plt.figure()
sns.lineplot(avg_df, x="time_point", y="aspect_ratio_avg", hue="size")
plt.show()
plt.figure()
sns.lineplot(local_df, x="time_point", y="S_value", hue="size")
plt.show()
plt.figure()
sns.lineplot(avg_df, x="time_point", y="S_avg", hue="size")
plt.show()

In [ ]:
plot_violin_with_swarm(local_df, y_col="theta", show_swarm=False)

In [ ]:
local_df["z_centered"] = local_df.groupby(["time_point", "size", "gastruloid"])["slice_z"].transform(
    lambda x: x - np.median(x))

In [ ]:
z_min = 0.0
z_max = 1.0
zcroplabel = f" cropped in z by {100 * z_min:.2f} - {100 * z_max:.2f}%"
local_df_cropped_z = crop_z_fraction(local_df, z_min_frac=z_min, z_max_frac=z_max)

plt.figure(figsize=(12, 6))
sns.lineplot(data=local_df_cropped_z, x="z_centered", y="S_value", hue="time_point", style="size",
             marker="o", alpha=0.7, palette="coolwarm")
plt.show()

# HERE

In [ ]:
def crop_z_by_range(df, z_range):
    """
    Keep slices within ±z_range around the middle slice (closest to z_centered = 0)
    for each (time_point, size, gastruloid).
    """
    df = df.copy()

    # Find the middle z slice per gastruloid: closest to 0
    mid = (
        df.assign(abs_z=lambda d: d["z_centered"].abs())
        .sort_values("abs_z")
        .groupby(["time_point", "size", "gastruloid"])
        .first()["z_centered"]
        .rename("z_middle")
    )

    # Merge back
    df = df.merge(mid, on=["time_point", "size", "gastruloid"], how="left")

    # Keep only slices within ± z_range
    df = df[(df["z_centered"] >= df["z_middle"] - z_range) &
            (df["z_centered"] <= df["z_middle"] + z_range)]

    return df

In [ ]:
local_df_crop = crop_z_by_range(local_df, z_range=15)
sns.lineplot(
    data=local_df_crop,
    x="z_centered",
    y="aspect_ratio",
    hue="time_point",
    style="size",
    lw=1.2,
    alpha=0.6,
    palette="coolwarm"
)
plt.show()

In [ ]:
avg_df["z_centered"] = avg_df.groupby(["time_point", "size", "gastruloid"])["slice_z"].transform(
    lambda x: x - np.median(x))
avg_df_crop = crop_z_by_range(avg_df, z_range=11)
sns.lineplot(
    data=avg_df_crop,
    x="z_centered",
    y="aspect_ratio_avg",
    hue="time_point",
    style="size",
    lw=1.2,
    alpha=0.6,
    palette="coolwarm"
)
plt.show()

In [ ]:
plot_violin_with_swarm(avg_df_crop, 'aspect_ratio_avg', title="Average Aspect Ratio per Slice", palette='Set1',
                       show_swarm=True)

In [ ]:
local_df_crop = crop_z_by_range(local_df, z_range=45)
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=local_df_crop,
    x="z_centered",
    y="S_value",
    hue="time_point",
    style="size",
    lw=1.2,
    alpha=0.6,
    palette="coolwarm"
)
plt.show()

In [ ]:
y_val = "aspect_ratio"
local_df_crop = crop_z_by_range(local_df, z_range=15)
sns.lineplot(
    data=local_df_crop,
    x="z_centered",
    y=y_val,
    hue="time_point",
    style="size",
    lw=1.2,
    alpha=0.6,
    palette="coolwarm"
)
plt.show()

# Average aspect ratio per gastruloid across all z slices
df_gastruloid_avg = (
    local_df_crop.groupby(["time_point", "size", "gastruloid"], as_index=False)
    [y_val]
    .mean()
    .rename(columns={y_val: f"{y_val}_avg"})
)
plot_violin_with_swarm(
    df_gastruloid_avg,
    y_col=f"{y_val}_avg",
    title="Average Aspect Ratio per Gastruloid",
    palette='Set1',
    show_swarm=True
)

# Average aspect ratio per gastruloid across all z slices
df_gastruloid_slice_avg = (
    local_df_crop.groupby(["time_point", "size", "gastruloid", "slice_z"], as_index=False)
    [y_val]
    .mean()
    .rename(columns={y_val: f"{y_val}_avg"})
)
plot_violin_with_swarm(
    df_gastruloid_slice_avg,
    y_col=f"{y_val}_avg",
    title="Average Aspect Ratio within each Gastruloid",
    palette='Set1',
    show_swarm=True
)

In [ ]:
y_val = "S_value"
local_df_crop = crop_z_by_range(local_df, z_range=15)
sns.lineplot(
    data=local_df_crop,
    x="z_centered",
    y=y_val,
    hue="time_point",
    style="size",
    lw=1.2,
    alpha=0.6,
    palette="coolwarm"
)
plt.show()

# Average aspect ratio per gastruloid across all z slices
df_gastruloid_avg = (
    local_df_crop.groupby(["time_point", "size", "gastruloid"], as_index=False)
    [y_val]
    .mean()
    .rename(columns={y_val: f"{y_val}_avg"})
)
plot_violin_with_swarm(
    df_gastruloid_avg,
    y_col=f"{y_val}_avg",
    title="Average Aspect Ratio per Gastruloid",
    palette='Set1',
    show_swarm=True
)

# Average aspect ratio per gastruloid across all z slices
df_gastruloid_slice_avg = (
    local_df_crop.groupby(["time_point", "size", "gastruloid", "slice_z"], as_index=False)
    [y_val]
    .mean()
    .rename(columns={y_val: f"{y_val}_avg"})
)
plot_violin_with_swarm(
    df_gastruloid_slice_avg,
    y_col=f"{y_val}_avg",
    title="Average Aspect Ratio within each Gastruloid",
    palette='Set1',
    show_swarm=True
)

In [ ]:
z_min = 0.0
z_max = 0.25
zcroplabel = f" cropped in z by {100 * z_min:.2f} - {100 * z_max:.2f}%"
local_df_cropped_z = crop_z_fraction(local_df, z_min_frac=z_min, z_max_frac=z_max)
avg_df_cropped_z = crop_z_fraction(avg_df, z_min_frac=z_min, z_max_frac=z_max)
# plot_violin_with_swarm(local_df_cropped_z, 'aspect_ratio', title="Aspect ratio with all per slices"+zcroplabel, palette='Set1')
plot_violin_with_swarm(avg_df_cropped_z, 'aspect_ratio_avg', title="Aspect ratio with average per slice" + zcroplabel,
                       palette='Set1', show_swarm=True)
z_min = 0.25
z_max = 0.5
zcroplabel = f" cropped in z by {100 * z_min:.2f} - {100 * z_max:.2f}%"
local_df_cropped_z = crop_z_fraction(local_df, z_min_frac=z_min, z_max_frac=z_max)
avg_df_cropped_z = crop_z_fraction(avg_df, z_min_frac=z_min, z_max_frac=z_max)
# plot_violin_with_swarm(local_df_cropped_z, 'aspect_ratio', title="Aspect ratio with all per slices"+zcroplabel, palette='Set1')
plot_violin_with_swarm(avg_df_cropped_z, 'aspect_ratio_avg', title="Aspect ratio with average per slice" + zcroplabel,
                       palette='Set1', show_swarm=True)
z_min = 0.5
z_max = 0.75
zcroplabel = f" cropped in z by {100 * z_min:.2f} - {100 * z_max:.2f}%"
local_df_cropped_z = crop_z_fraction(local_df, z_min_frac=z_min, z_max_frac=z_max)
avg_df_cropped_z = crop_z_fraction(avg_df, z_min_frac=z_min, z_max_frac=z_max)
# plot_violin_with_swarm(local_df_cropped_z, 'aspect_ratio', title="Aspect ratio with all per slices"+zcroplabel, palette='Set1')
plot_violin_with_swarm(avg_df_cropped_z, 'aspect_ratio_avg', title="Aspect ratio with average per slice" + zcroplabel,
                       palette='Set1', show_swarm=True)
z_min = 0.75
z_max = 1.0
zcroplabel = f" cropped in z by {100 * z_min:.2f} - {100 * z_max:.2f}%"
local_df_cropped_z = crop_z_fraction(local_df, z_min_frac=z_min, z_max_frac=z_max)
avg_df_cropped_z = crop_z_fraction(avg_df, z_min_frac=z_min, z_max_frac=z_max)
# plot_violin_with_swarm(local_df_cropped_z, 'aspect_ratio', title="Aspect ratio with all per slices"+zcroplabel, palette='Set1')
plot_violin_with_swarm(avg_df_cropped_z, 'aspect_ratio_avg', title="Aspect ratio with average per slice" + zcroplabel,
                       palette='Set1', show_swarm=True)

In [ ]:
z_min = 0.25
z_max = 0.75
zcroplabel = f" cropped in z by {100 * z_min:.2f} - {100 * z_max:.2f}%"
local_df_cropped_z = crop_z_fraction(local_df, z_min_frac=z_min, z_max_frac=z_max)
avg_df_cropped_z = crop_z_fraction(avg_df, z_min_frac=z_min, z_max_frac=z_max)
plot_violin_with_swarm(avg_df_cropped_z, 'aspect_ratio_avg', title="Aspect ratio with average per slice" + zcroplabel,
                       palette='Set1', show_swarm=True, swarm_size=2)

In [ ]:
plot_jaja(df=local_df)
z_min = 0.25
z_max = 0.75
local_df_cropped_z = crop_z_fraction(local_df, z_min_frac=z_min, z_max_frac=z_max)
plot_jaja(df=local_df_cropped_z)

In [ ]:
# ---- Choose condition ----
chosen_time = "104"
chosen_size = "200"
chosen_gastruloid = "Gas1"  # pick one gastruloid
weighted = True  # True for S_value, False for S_value_norm

subset = local_df[
    (local_df["time_point"] == chosen_time) &
    (local_df["size"] == chosen_size) &
    (local_df["gastruloid"] == chosen_gastruloid) &
    (local_df["z_centered"] == 0)
    ]

if subset.empty:
    print(f"No data found for {chosen_time}h, size {chosen_size}, gastruloid {chosen_gastruloid}")
else:
    # choose which S column
    S_col = "S_value" if weighted else "S_value_norm"

    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(
        subset["xpos"], subset["ypos"], C=subset[S_col], gridsize=50,
        reduce_C_function=np.mean, cmap="viridis"
    )
    plt.colorbar(hb, label=f"Weighted local order ({S_col})")
    plt.xlabel("X position (µm)")
    plt.ylabel("Y position (µm)")
    plt.title(f"2D spatial map of local order — {chosen_gastruloid}, {chosen_size}, {chosen_time}h")
    plt.axis("equal")
    plt.show()

## Overview of Dataset Statistics

In [ ]:
# ---- Step 1: count gastruloids per time point and size ----
gastruloid_counts = (
    avg_df.groupby(['time_point', 'size'])['gastruloid']
    .nunique()
    .reset_index()  # reset_index first
)

# Rename the new column
gastruloid_counts.rename(columns={'gastruloid': 'n_gastruloids'}, inplace=True)

# Ensure time points are ordered numerically
gastruloid_counts['time_point'] = pd.Categorical(
    gastruloid_counts['time_point'],
    categories=sorted(gastruloid_counts['time_point'].unique(), key=int),
    ordered=True
)

# ---- Step 2: plot ----
plt.figure(figsize=(10, 6))
sns.barplot(
    data=gastruloid_counts,
    x='time_point',
    y='n_gastruloids',
    hue='size',
    palette='Set2'
)

plt.xlabel("Time point")
plt.ylabel("Number of gastruloids")
plt.title("Number of gastruloids per time point, grouped by seeding size")
plt.grid(alpha=0.3, axis='y')
plt.legend(title='Size')
plt.savefig(os.path.join(comp_fig_dir, f'overall-count_statistics.{comp_fig_type}'), dpi=comp_fig_dpi)
plt.show()

In [ ]:
# ---- Step 1: number of gastruloids per time × size ----
gastruloid_counts = (
    avg_df.groupby(['time_point', 'size'])['gastruloid']
    .nunique()
    .reset_index()
    .rename(columns={'gastruloid': 'n_gastruloids'})
)

# ---- Step 2: number of z-slices per gastruloid ----
z_slices = (
    local_df.groupby(['time_point', 'size', 'gastruloid'])['slice_z']
    .nunique()
    .reset_index()
    .rename(columns={'slice_z': 'n_slices'})
)

z_slice_stats = (
    z_slices.groupby(['time_point', 'size'])['n_slices']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'mean_slices', 'std': 'std_slices'})
)

# ---- Step 3: number of cells per slice ----
cells_per_slice = (
    local_df.groupby(['time_point', 'size', 'gastruloid', 'slice_z'])
    .size()
    .reset_index(name='n_cells')
)

# ---- Step 4: restrict to middle z-section (40–60%) ----


mid_cells = (
    cells_per_slice.groupby(['time_point', 'size', 'gastruloid'])
    .apply(middle_z_mean)
    .reset_index(name='mean_cells_mid_z')
)

cell_stats = (
    mid_cells.groupby(['time_point', 'size'])['mean_cells_mid_z']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'mean_cells_mid_z', 'std': 'std_cells_mid_z'})
)

# ---- Step 5: combine all ----
summary_df = (
    gastruloid_counts
    .merge(z_slice_stats, on=['time_point', 'size'], how='left')
    .merge(cell_stats, on=['time_point', 'size'], how='left')
)

# ---- Step 6: ensure time_point is numeric and sorted ----
summary_df["time_point"] = summary_df["time_point"].astype(int)
summary_df = summary_df.sort_values(by=["size", "time_point"])

# ---- Step 7: plot ----
fig, axes = plt.subplots(3, 1, figsize=(10, 14), sharex=True)

#  common order
time_order = sorted(summary_df["time_point"].unique())

# 1️⃣ Gastruloid counts
sns.barplot(
    data=summary_df,
    x='time_point', y='n_gastruloids', hue='size',
    order=time_order, palette='Set2', ax=axes[0]
)
axes[0].set_title("Number of gastruloids per time point and size")
axes[0].set_ylabel("n gastruloids")
axes[0].grid(alpha=0.3, axis='y')

# 2️⃣ Mean number of z-slices per gastruloid
sns.barplot(
    data=summary_df,
    x='time_point', y='mean_slices', hue='size',
    order=time_order, palette='Set2', ax=axes[1]
)
axes[1].set_title("Mean number of z-slices per gastruloid")
axes[1].set_ylabel("Mean n slices")
axes[1].grid(alpha=0.3, axis='y')

# 3️⃣ Mean number of cells per slice (middle z only)
sns.barplot(
    data=summary_df,
    x='time_point', y='mean_cells_mid_z', hue='size',
    order=time_order, palette='Set2', ax=axes[2]
)
axes[2].set_title("Mean number of cells per slice (middle z-section)")
axes[2].set_ylabel("Mean n cells (40–60% z-range)")
axes[2].grid(alpha=0.3, axis='y')

plt.xlabel("Time point")
plt.tight_layout()
plt.savefig(
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/overview.png",
    dpi=300)
plt.show()

In [ ]:
plot_violin_with_swarm(local_df, 'aspect_ratio', title="Aspect ratio with all per slices", palette='Set1')
plot_violin_with_swarm(avg_df, 'aspect_ratio_avg', title="Aspect ratio with average per slice", palette='Set1')
plot_violin_with_swarm(local_df, 'S_value', title="Weighted Nematic Order S with all per slices", palette='Set2')
plot_violin_with_swarm(avg_df, 'S_avg', title="Weighted Nematic Order S with average per slice", palette='Set2')

In [ ]:
plot_violin_with_swarm(local_df, 'radius1', title="Ellipse Longest Axis", palette='pastel')
plot_violin_with_swarm(local_df, 'radius2', title="Ellipse Longest Axis", palette='pastel')

In [ ]:
local_df["ellipse_area"] = np.pi * local_df["radius1"] * local_df["radius2"]
plot_violin_with_swarm(local_df, 'ellipse_area', title="Ellipse Area", palette='pastel')
avg_df["ellipse_area_avg"] = np.pi * avg_df["radius1_avg"] * avg_df["radius2_avg"]
plot_violin_with_swarm(avg_df, 'ellipse_area_avg', title="Ellipse Area", palette='pastel')

In [ ]:
# Choose measurement
y_col = "S_value"
# y_col = "aspect_ratio"

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=local_df,
    x="slice_z",
    y=y_col,
    hue="time_point",
    style="size",
    ci="sd",  # shaded area = standard deviation
    markers=True,
    dashes=False
)
plt.xlabel("Slice index z")
plt.ylabel(y_col.replace('_', ' ').capitalize())
plt.title(f"{y_col.replace('_', ' ').capitalize()} vs slice index z")
plt.grid(alpha=0.3)
plt.savefig(os.path.join(comp_fig_dir, f'{y_col}.{comp_fig_type}'), dpi=comp_fig_dpi)
plt.show()

In [ ]:
local_df["z_centered"] = (
    local_df.groupby(["time_point", "size", "gastruloid"])["slice_z"]
    .transform(lambda x: x - np.median(x))
)

In [ ]:
t_sel, size_sel = "104", "200"
plt.title(f"Time point = {t_sel}h and Size = {size_sel}")
sns.histplot(data=local_df[(local_df["time_point"] == t_sel) & (local_df["size"] == size_sel)], y="z_centered",
             x="aspect_ratio", bins=20, cmap="viridis")

In [ ]:
# Create quantile bins and capture bin edges
z_bins, bin_edges = pd.qcut(local_df["z_centered"], q=5, retbins=True, duplicates="drop")

# Format bin labels with ranges (rounded for clarity)
bin_labels = [
    f"{bin_edges[i]:.1f}–{bin_edges[i + 1]:.1f}" for i in range(len(bin_edges) - 1)
]

# Apply categorical labels
local_df["z_group"] = pd.qcut(local_df["z_centered"], q=5, labels=bin_labels)

# Plot
plt.figure(figsize=(12, 6))
sns.violinplot(
    data=local_df[local_df["size"] == "200"],
    x="time_point",
    y="aspect_ratio",
    hue="z_group",
    palette="Set1",
    inner="quartile"
)
plt.legend(title="z centered range", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.title("Aspect ratio vs time (z binned by quantile range)")
plt.tight_layout()
plt.show()

# Create quantile bins and capture bin edges
z_bins, bin_edges = pd.qcut(local_df["z_centered"], q=5, retbins=True, duplicates="drop")

# Format bin labels with ranges (rounded for clarity)
bin_labels = [
    f"{bin_edges[i]:.1f}–{bin_edges[i + 1]:.1f}" for i in range(len(bin_edges) - 1)
]

# Apply categorical labels
local_df["z_group"] = pd.qcut(local_df["z_centered"], q=5, labels=bin_labels)

# Plot
plt.figure(figsize=(12, 6))
sns.violinplot(
    data=local_df[local_df["size"] == "200"],
    x="time_point",
    y="S_value",
    hue="z_group",
    palette="Set2",
    inner="quartile"
)
plt.legend(title="z centered range", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.title("S_value vs time (z binned by quantile range)")
plt.tight_layout()
plt.show()

In [ ]:
# Average S across all gastruloids, all time points
plot_aligned_z(avg_df, y_col="S_avg")

# Local weighted S per cell
plot_aligned_z(local_df, y_col="S_value")

# Aspect ratio per slice, all time points
plot_aligned_z(avg_df, y_col="aspect_ratio_avg")
plot_aligned_z(avg_df, y_col="S_avg", time_point="104", style_col="gastruloid")

In [ ]:
plot_aligned_z(avg_df, y_col="S_avg", time_point="104", style_col="gastruloid", separate_gastruloids=True, size=200)

In [ ]:
plot_aligned_z(avg_df, y_col="aspect_ratio_avg", size="200", mode="line", separate_gastruloids=True)

In [ ]:
sns.scatterplot(local_df[local_df["size"] == "200"], x="aspect_ratio", y="S_value", hue="time_point", alpha=0.5)

In [ ]:
sns.histplot(
    local_df[(local_df["time_point"] == "72") & (local_df["size"] == "200")],
    x="aspect_ratio", y="S_value_norm", hue="time_point")
plt.show()
sns.histplot(
    local_df[(local_df["time_point"] == "120") & (local_df["size"] == "200")],
    x="aspect_ratio", y="S_value_norm", hue="time_point")

In [ ]:
local_df_plot = local_df.copy()
local_df_plot["size_numeric"] = local_df_plot["size"].astype(float)  # convert to numeric

plt.figure(figsize=(10, 10))
sns.scatterplot(
    data=local_df_plot,
    x="aspect_ratio",
    y="S_value",
    hue="time_point",
    size="size_numeric",
    alpha=0.5,
    palette="coolwarm",
    sizes=(20, 100)  # optionally control min/max marker sizes
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
sns.histplot(local_df, x="aspect_ratio", y="S_value", alpha=0.9,
             palette="coolwarm")
plt.title("Weighted nematic order $S$ using 6 nearest-neighbours vs. aspect ratio")